In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
df = pd.read_csv("qoute_dataset.csv")

In [5]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [6]:
df['quote'][0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [7]:
df.shape

(3038, 2)

In [8]:
quotes = df['quote']
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [9]:
quotes = quotes.str.lower()

In [10]:
import string
translator = str.maketrans('','',string.punctuation)
quotes = quotes.apply(lambda x : x.translate(translator))

In [11]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


In [12]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [13]:
vocab_size = 10000
tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)

In [14]:
word_index = tokenizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [15]:
sequence = tokenizer.texts_to_sequences(quotes)

In [16]:
quotes[0]

'“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”'

In [17]:
print(sequence[0])

[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]


In [18]:
X = []
y = []

for seq in sequence:
    for i in range(1,len(seq)):
        input_seq = seq[:i]
        output_seq = seq[i]
        X.append(input_seq)
        y.append(output_seq)


In [19]:
len(X)

85271

In [20]:
len(y)

85271

In [21]:
max_len = max(len(x) for x in X)
print(max_len)

745


In [22]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(X,maxlen = max_len,padding = 'pre')

In [23]:
X_padded[0]

array([  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   

In [24]:
y = np.array(y)

In [25]:
X_padded.shape

(85271, 745)

In [26]:
y.shape

(85271,)

In [27]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y,num_classes = vocab_size)

In [28]:
y.shape

(85271,)

In [29]:
y_one_hot.shape

(85271, 10000)

In [30]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,SimpleRNN

In [31]:
embedding_dim = 50
rnn_units = 128

In [32]:
rnn_model = Sequential()
rnn_model.add(
    Embedding(input_dim=vocab_size,output_dim=embedding_dim,input_length=max_len)
)

rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size,activation='softmax'))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [33]:
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics= ['accuracy']
)

In [34]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [35]:
lstm_model = Sequential()

In [36]:
lstm_model.add(
    Embedding(input_dim=vocab_size,output_dim=embedding_dim,input_length=max_len)
)
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size,activation='softmax'))

In [37]:
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics= ['accuracy']
)

In [38]:
epochs =100
batch_size = 128

In [39]:
history_rnn = rnn_model.fit(
    X_padded,y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1
)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 66s 94ms/step - accuracy: 0.0457 - loss: 6.7093 - val_accuracy: 0.0595 - val_loss: 6.5376
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 44s 73ms/step - accuracy: 0.0793 - loss: 6.0822 - val_accuracy: 0.0902 - val_loss: 6.3369
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 44s 72ms/step - accuracy: 0.1037 - loss: 5.7302 - val_accuracy: 0.1013 - val_loss: 6.3015
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 44s 73ms/step - accuracy: 0.1195 - loss: 5.4393 - val_accuracy: 0.1057 - val_loss: 6.3410
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 44s 72ms/step - accuracy: 0.1315 - loss: 5.2015 - val_accuracy: 0.1067 - val_loss: 6.3654
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 44s 73ms/step - accuracy: 0.1421 - loss: 5.0089 - val_accuracy: 0.1110 - val_loss: 6.3575
Epoch 7/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 44s 73ms/step - accuracy: 0.1563 - loss: 4.7713 - val_accuracy: 0.1103 - val_loss: 6.4326
Epoch 8/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 44s 73ms/step - accuracy: 0.1711 - loss: 4

In [40]:
history_lstm = lstm_model.fit(
    X_padded,y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1
)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 24s 34ms/step - accuracy: 0.0391 - loss: 6.7571 - val_accuracy: 0.0462 - val_loss: 6.6833
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 20s 33ms/step - accuracy: 0.0585 - loss: 6.3228 - val_accuracy: 0.0629 - val_loss: 6.5658
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 20s 33ms/step - accuracy: 0.0790 - loss: 6.0582 - val_accuracy: 0.0869 - val_loss: 6.4731
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 20s 33ms/step - accuracy: 0.0980 - loss: 5.8299 - val_accuracy: 0.0976 - val_loss: 6.4381
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 20s 33ms/step - accuracy: 0.1097 - loss: 5.6421 - val_accuracy: 0.1014 - val_loss: 6.4128
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 20s 33ms/step - accuracy: 0.1190 - loss: 5.4696 - val_accuracy: 0.1062 - val_loss: 6.4204
Epoch 7/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 20s 33ms/step - accuracy: 0.1288 - loss: 5.3112 - val_accuracy: 0.1072 - val_loss: 6.4435
Epoch 8/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 20s 33ms/step - accuracy: 0.1355 - loss: 5

In [41]:
lstm_model.save("lstm_model.h5")

In [42]:
rnn_model.save("rnn_model.h5")

In [43]:
index_to_word ={}
for word,index in word_index.items():
  index_to_word[index] = word

In [46]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [49]:
def predictor(model,tokenizer,text,max_len):
  text = text.lower()
  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq],maxlen=max_len,padding='pre')
  pred = model.predict(seq,verbose=0)
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]


In [52]:
seed_text = "what are you"
next_word = predictor(lstm_model,tokenizer,seed_text,max_len)
print(next_word)

implying


In [54]:
def generate_text(model,tokenizer,seed_text,n_words,max_len):
  for _ in range(n_words):
    next_word = predictor(model,tokenizer,seed_text,max_len)
    if next_word == "":
      break
    seed_text += " "+next_word
  return seed_text

In [55]:
seed = "the meaning of life"
generate_text = generate_text(lstm_model,tokenizer,seed,10,max_len)
print(generate_text)

the meaning of life is not in love it is not madness it is


In [56]:
import pickle
with open('tokenizer.pkl','wb') as f:
  pickle.dump(tokenizer,f)

In [57]:
with open("max_len.pkl","wb") as f:
  pickle.dump(max_len,f)